In [ ]:
# =========================================
# STEP 1: Install Libraries
# =========================================
!pip install -q transformers datasets peft accelerate bitsandbytes trl nltk scikit-learn

# =========================================
# STEP 2: Imports
# =========================================
import torch
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np

nltk.download('punkt')
nltk.download('punkt_tab')

# =========================================
# STEP 3: LOAD YOUR DATASET
# =========================================
with open("data.json", "r") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)
print(f"✅ Loaded {len(dataset)} samples")

# =========================================
# STEP 4: FORMAT DATA
# =========================================
def format_prompt(sample):
    return {
        "text": f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['output']}"
    }

formatted_dataset = dataset.map(format_prompt)

# =========================================
# STEP 5: LOAD MODEL
# =========================================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False
print("✅ Model Loaded")

# =========================================
# STEP 6: LoRA CONFIG (UPGRADED)
# =========================================
lora_config = LoraConfig(
    r=64, # Increased LoRA rank
    lora_alpha=128, # Increased LoRA alpha
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Expanded target modules
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("✅ LoRA Applied")

# =========================================
# STEP 7: TOKENIZATION
# =========================================
def tokenize_function(sample):
    return tokenizer(
        sample["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
print("✅ Dataset Tokenized")

# =========================================
# STEP 8: TRAINING CONFIG
# =========================================
training_arguments = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=10,   # 🔥 increased epochs
    learning_rate=1e-4, # 🔥 adjusted learning rate
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# =========================================
# STEP 9: TRAIN MODEL
# =========================================
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_arguments,
)

print("🚀 Training Starting...")
trainer.train()
print("✅ Training Complete")

# =========================================
# STEP 10: SAVE MODEL
# =========================================
trainer.save_model("fine-tuned-model")
tokenizer.save_pretrained("fine-tuned-model")
print("✅ Model Saved")

# =========================================
# STEP 11: INFERENCE (CLEAN VERSION)
# =========================================
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "fine-tuned-model")

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def generate_response(question):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    result = generator(
        prompt,
        max_new_tokens=40,
        do_sample=False
    )
    return result[0]["generated_text"][len(prompt):].strip()

# Test
print("\n🧪 Sample Output:")
print(generate_response("What is Artificial Intelligence?"))

# =========================================
# STEP 12: IMPROVED EVALUATION
# =========================================
eval_data = data[:10]

references = [x["output"] for x in eval_data]
predictions = []

for item in eval_data:
    pred = generate_response(item["instruction"])
    predictions.append(pred)

# 🔥 Relaxed Accuracy (Better for LLMs)
def relaxed_match(refs, preds):
    correct = 0
    for r, p in zip(refs, preds):
        if r.lower() in p.lower() or p.lower() in r.lower():
            correct += 1
    return correct / len(refs)

# BLEU Score
def bleu(refs, preds):
    smoothie = SmoothingFunction().method1
    scores = []
    for r, p in zip(refs, preds):
        scores.append(sentence_bleu([nltk.word_tokenize(r)], nltk.word_tokenize(p), smoothing_function=smoothie))
    return np.mean(scores)

print("\n📊 FINAL Evaluation:")
print("Relaxed Accuracy:", relaxed_match(references, predictions))
print("BLEU Score:", bleu(references, predictions))

In [ ]:
# =========================================
# STEP 1: Install Libraries
# =========================================
!pip install -q transformers datasets peft accelerate bitsandbytes trl nltk scikit-learn

# =========================================
# STEP 2: Imports
# =========================================
import torch
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np

nltk.download('punkt')
nltk.download('punkt_tab')

# =========================================
# STEP 3: LOAD DATASET
# =========================================
with open("data.json", "r") as f:
    data = json.load(f)

print(f"✅ Total Samples: {len(data)}")

# =========================================
# STEP 4: TRAIN / TEST SPLIT ✅
# =========================================
split_index = int(0.8 * len(data))

train_data = data[:split_index]
test_data = data[split_index:]

print(f"✅ Train: {len(train_data)} | Test: {len(test_data)}")

train_dataset = Dataset.from_list(train_data)

# =========================================
# STEP 5: FORMAT DATA
# =========================================
def format_prompt(sample):
    return {
        "text": f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['output']}"
    }

formatted_dataset = train_dataset.map(format_prompt)

# =========================================
# STEP 6: LOAD MODEL
# =========================================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False
print("✅ Model Loaded")

# =========================================
# STEP 7: LoRA CONFIG
# =========================================
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("✅ LoRA Applied")

# =========================================
# STEP 8: TOKENIZATION
# =========================================
def tokenize_function(sample):
    return tokenizer(
        sample["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
print("✅ Dataset Tokenized")

# =========================================
# STEP 9: TRAINING CONFIG
# =========================================
training_arguments = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=6,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# =========================================
# STEP 10: TRAIN MODEL
# =========================================
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_arguments,
)

print("🚀 Training Starting...")
trainer.train()
print("✅ Training Complete")

# =========================================
# STEP 11: SAVE MODEL
# =========================================
trainer.save_model("fine-tuned-model")
tokenizer.save_pretrained("fine-tuned-model")
print("✅ Model Saved")

# =========================================
# STEP 12: INFERENCE
# =========================================
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "fine-tuned-model")

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def generate_response(question):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    result = generator(
        prompt,
        max_new_tokens=35,
        do_sample=False
    )
    return result[0]["generated_text"][len(prompt):].strip()

# =========================================
# STEP 13: REAL EVALUATION (TEST DATA ONLY)
# =========================================
eval_data = test_data

references = [x["output"] for x in eval_data]
predictions = []

for item in eval_data:
    pred = generate_response(item["instruction"])
    predictions.append(pred)

# Relaxed Accuracy
def relaxed_match(refs, preds):
    correct = 0
    for r, p in zip(refs, preds):
        if r.lower() in p.lower() or p.lower() in r.lower():
            correct += 1
    return correct / len(refs)

# BLEU Score
def bleu(refs, preds):
    smoothie = SmoothingFunction().method1
    scores = []
    for r, p in zip(refs, preds):
        scores.append(sentence_bleu([nltk.word_tokenize(r)], nltk.word_tokenize(p), smoothing_function=smoothie))
    return np.mean(scores)

print("\n📊 FINAL Evaluation (REAL):")
print("Relaxed Accuracy:", relaxed_match(references, predictions))
print("BLEU Score:", bleu(references, predictions))

# =========================================
# STEP 14: SHOW SAMPLE OUTPUTS
# =========================================
print("\n🧪 Sample Predictions:")
for i in range(min(3, len(eval_data))):
    print("\nQ:", eval_data[i]["instruction"])
    print("Expected:", eval_data[i]["output"])
    print("Predicted:", predictions[i])

In [ ]:
# =========================================
# STEP 1: Install Libraries
# =========================================
!pip install -q transformers datasets peft accelerate bitsandbytes trl nltk scikit-learn

# =========================================
# STEP 2: Imports
# =========================================
import torch
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np

nltk.download('punkt')
nltk.download('punkt_tab')

# =========================================
# STEP 3: LOAD DATASET
# =========================================
with open("data.json", "r") as f:
    data = json.load(f)

print(f"✅ Total Samples: {len(data)}")

# =========================================
# STEP 4: TRAIN / TEST SPLIT ✅
# =========================================
split_index = int(0.8 * len(data))

train_data = data[:split_index]
test_data = data[split_index:]

print(f"✅ Train: {len(train_data)} | Test: {len(test_data)}")

train_dataset = Dataset.from_list(train_data)

# =========================================
# STEP 5: FORMAT DATA
# =========================================
def format_prompt(sample):
    return {
        "text": f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['output']}"
    }

formatted_dataset = train_dataset.map(format_prompt)

# =========================================
# STEP 6: LOAD MODEL
# =========================================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False
print("✅ Model Loaded")

# =========================================
# STEP 7: LoRA CONFIG
# =========================================
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("✅ LoRA Applied")

# =========================================
# STEP 8: TOKENIZATION
# =========================================
def tokenize_function(sample):
    return tokenizer(
        sample["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
print("✅ Dataset Tokenized")

# =========================================
# STEP 9: TRAINING CONFIG
# =========================================
training_arguments = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=6,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# =========================================
# STEP 10: TRAIN MODEL
# =========================================
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_arguments,
)

print("🚀 Training Starting...")
trainer.train()
print("✅ Training Complete")

# =========================================
# STEP 11: SAVE MODEL
# =========================================
trainer.save_model("fine-tuned-model")
tokenizer.save_pretrained("fine-tuned-model")
print("✅ Model Saved")

# =========================================
# STEP 12: INFERENCE
# =========================================
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "fine-tuned-model")

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def generate_response(question):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    result = generator(
        prompt,
        max_new_tokens=35,
        do_sample=False
    )
    return result[0]["generated_text"][len(prompt):].strip()

# =========================================
# STEP 13: IMPROVED REAL EVALUATION
# =========================================
from difflib import SequenceMatcher

eval_data = test_data

references = [x["output"] for x in eval_data]
predictions = []

for item in eval_data:
    pred = generate_response(item["instruction"])
    predictions.append(pred)

# ❌ Old relaxed match (kept for comparison)
def relaxed_match(refs, preds):
    correct = 0
    for r, p in zip(refs, preds):
        if r.lower() in p.lower() or p.lower() in r.lower():
            correct += 1
    return correct / len(refs)

# ✅ NEW: Semantic Accuracy (BEST METRIC)
def semantic_accuracy(refs, preds):
    scores = []
    for r, p in zip(refs, preds):
        score = SequenceMatcher(None, r.lower(), p.lower()).ratio()
        scores.append(score)
    return sum(scores) / len(scores)

# BLEU Score (keep)
def bleu(refs, preds):
    smoothie = SmoothingFunction().method1
    scores = []
    for r, p in zip(refs, preds):
        scores.append(
            sentence_bleu(
                [nltk.word_tokenize(r)],
                nltk.word_tokenize(p),
                smoothing_function=smoothie
            )
        )
    return np.mean(scores)

print("\n📊 FINAL Evaluation (REAL):")
print("Relaxed Accuracy:", relaxed_match(references, predictions))
print("Semantic Accuracy:", semantic_accuracy(references, predictions))  # ⭐ NEW
print("BLEU Score:", bleu(references, predictions))
# =========================================
# STEP 14: SHOW SAMPLE OUTPUTS
# =========================================
print("\n🧪 Sample Predictions:")
for i in range(min(3, len(eval_data))):
    print("\nQ:", eval_data[i]["instruction"])
    print("Expected:", eval_data[i]["output"])
    print("Predicted:", predictions[i])

In [ ]:
# =========================================
# STEP 1: Install Libraries
# =========================================
!pip install -q transformers datasets peft accelerate bitsandbytes trl nltk scikit-learn

# =========================================
# STEP 2: Imports
# =========================================
import torch
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np

nltk.download('punkt')
nltk.download('punkt_tab')

# =========================================
# STEP 3: LOAD DATASET
# =========================================
with open("data.json", "r") as f:
    data = json.load(f)

print(f"✅ Total Samples: {len(data)}")

# =========================================
# STEP 4: TRAIN / TEST SPLIT ✅
# =========================================
split_index = int(0.8 * len(data))

train_data = data[:split_index]
test_data = data[split_index:]

print(f"✅ Train: {len(train_data)} | Test: {len(test_data)}")

train_dataset = Dataset.from_list(train_data)

# =========================================
# STEP 5: FORMAT DATA
# =========================================
def format_prompt(sample):
    return {
        "text": f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['output']}"
    }

formatted_dataset = train_dataset.map(format_prompt)

# =========================================
# STEP 6: LOAD MODEL
# =========================================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False
print("✅ Model Loaded")

# =========================================
# STEP 7: LoRA CONFIG
# =========================================
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("✅ LoRA Applied")

# =========================================
# STEP 8: TOKENIZATION
# =========================================
def tokenize_function(sample):
    return tokenizer(
        sample["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
print("✅ Dataset Tokenized")

# =========================================
# STEP 9: TRAINING CONFIG
# =========================================
training_arguments = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=6,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# =========================================
# STEP 10: TRAIN MODEL
# =========================================
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_arguments,
)

print("🚀 Training Starting...")
trainer.train()
print("✅ Training Complete")

# =========================================
# STEP 11: SAVE MODEL
# =========================================
trainer.save_model("fine-tuned-model")
tokenizer.save_pretrained("fine-tuned-model")
print("✅ Model Saved")

# =========================================
# STEP 12: INFERENCE
# =========================================
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "fine-tuned-model")

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def generate_response(question):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    result = generator(
        prompt,
        max_new_tokens=35,
        do_sample=False
    )
    return result[0]["generated_text"][len(prompt):].strip()

# =========================================
# STEP 13: FINAL HYBRID EVALUATION ✅
# =========================================
from difflib import SequenceMatcher

eval_data = test_data

references = [x["output"] for x in eval_data]
predictions = []

for item in eval_data:
    pred = generate_response(item["instruction"])
    predictions.append(pred)

# 🔥 1. Sequence similarity
def seq_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

# 🔥 2. Keyword overlap
def keyword_overlap(a, b):
    a_words = set(a.lower().split())
    b_words = set(b.lower().split())
    return len(a_words & b_words) / len(a_words)

# 🔥 3. FINAL HYBRID SCORE (BEST)
def hybrid_accuracy(refs, preds):
    scores = []
    for r, p in zip(refs, preds):
        sim_score = seq_similarity(r, p)
        overlap_score = keyword_overlap(r, p)

        # weighted average
        final_score = 0.5 * sim_score + 0.5 * overlap_score
        scores.append(final_score)

    return sum(scores) / len(scores)

# BLEU Score (keep for reference)
def bleu(refs, preds):
    smoothie = SmoothingFunction().method1
    scores = []
    for r, p in zip(refs, preds):
        scores.append(
            sentence_bleu(
                [nltk.word_tokenize(r)],
                nltk.word_tokenize(p),
                smoothing_function=smoothie
            )
        )
    return np.mean(scores)

print("\n📊 FINAL Evaluation (CORRECT):")
print("Hybrid Accuracy:", hybrid_accuracy(references, predictions))  # ⭐ FINAL METRIC
print("BLEU Score:", bleu(references, predictions))
# =========================================
# STEP 14: SHOW SAMPLE OUTPUTS
# =========================================
print("\n🧪 Sample Predictions:")
for i in range(min(3, len(eval_data))):
    print("\nQ:", eval_data[i]["instruction"])
    print("Expected:", eval_data[i]["output"])
    print("Predicted:", predictions[i])

In [ ]:
# =========================================
# STEP 1: Install Libraries
# =========================================
!pip install -q transformers datasets peft accelerate bitsandbytes trl nltk scikit-learn

# =========================================
# STEP 2: Imports
# =========================================
import torch
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
from difflib import SequenceMatcher

nltk.download('punkt')
nltk.download('punkt_tab')

# =========================================
# STEP 3: LOAD DATASET
# =========================================
with open("data.json", "r") as f:
    data = json.load(f)

print(f"✅ Total Samples: {len(data)}")

# =========================================
# STEP 4: TRAIN / TEST SPLIT
# =========================================
split_index = int(0.8 * len(data))

train_data = data[:split_index]
test_data = data[split_index:]

print(f"✅ Train: {len(train_data)} | Test: {len(test_data)}")

train_dataset = Dataset.from_list(train_data)

# =========================================
# STEP 5: FORMAT DATA
# =========================================
def format_prompt(sample):
    return {
        "text": f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['output']}"
    }

formatted_dataset = train_dataset.map(format_prompt)

# =========================================
# STEP 6: LOAD MODEL
# =========================================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False
print("✅ Model Loaded")

# =========================================
# STEP 7: LoRA CONFIG
# =========================================
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("✅ LoRA Applied")

# =========================================
# STEP 8: TOKENIZATION
# =========================================
def tokenize_function(sample):
    return tokenizer(
        sample["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
print("✅ Dataset Tokenized")

# =========================================
# STEP 9: TRAINING CONFIG
# =========================================
training_arguments = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=6,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# =========================================
# STEP 10: TRAIN MODEL
# =========================================
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_arguments,
)

print("🚀 Training Starting...")
trainer.train()
print("✅ Training Complete")

# =========================================
# STEP 11: SAVE MODEL
# =========================================
trainer.save_model("fine-tuned-model")
tokenizer.save_pretrained("fine-tuned-model")
print("✅ Model Saved")

# =========================================
# STEP 12: INFERENCE (UPDATED)
# =========================================
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "fine-tuned-model")

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def generate_response(question):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    result = generator(
        prompt,
        max_new_tokens=25,        # 🔥 shorter output
        do_sample=False,
        repetition_penalty=1.2    # 🔥 cleaner output
    )
    return result[0]["generated_text"][len(prompt):].strip()

# =========================================
# STEP 13: NORMALIZATION
# =========================================
def normalize(text):
    return text.lower().replace("the ", "").replace(".", "").strip()

# =========================================
# STEP 14: FINAL EVALUATION
# =========================================
eval_data = test_data

references = [normalize(x["output"]) for x in eval_data]
predictions = []

for item in eval_data:
    pred = generate_response(item["instruction"])
    predictions.append(normalize(pred))

# Sequence similarity
def seq_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# Keyword overlap
def keyword_overlap(a, b):
    a_words = set(a.split())
    b_words = set(b.split())
    return len(a_words & b_words) / len(a_words)

# Hybrid accuracy
def hybrid_accuracy(refs, preds):
    scores = []
    for r, p in zip(refs, preds):
        sim = seq_similarity(r, p)
        overlap = keyword_overlap(r, p)
        scores.append(0.5 * sim + 0.5 * overlap)
    return sum(scores) / len(scores)

# BLEU
def bleu(refs, preds):
    smoothie = SmoothingFunction().method1
    scores = []
    for r, p in zip(refs, preds):
        scores.append(
            sentence_bleu(
                [nltk.word_tokenize(r)],
                nltk.word_tokenize(p),
                smoothing_function=smoothie
            )
        )
    return np.mean(scores)

print("\n📊 FINAL Evaluation:")
print("Hybrid Accuracy:", hybrid_accuracy(references, predictions))
print("BLEU Score:", bleu(references, predictions))

# =========================================
# STEP 15: SAMPLE OUTPUTS
# =========================================
print("\n🧪 Sample Predictions:")
for i in range(min(3, len(eval_data))):
    print("\nQ:", eval_data[i]["instruction"])
    print("Expected:", eval_data[i]["output"])
    print("Predicted:", generate_response(eval_data[i]["instruction"]))